# PAPER1 · COLAB MASTER — one notebook: upload kaggle.json → download data → run pipeline

**How to use (3 steps):**
1. Runtime → Change runtime type → **GPU (T4)**. Run the SETUP cell, upload your `kaggle.json` when prompted.
2. In the DATASETS cell, flip the toggles for what to download (slugs are editable — verify with the search helper).
3. Set `STAGE` in the STAGE cell and run everything: `"week1_audio"` (inversion core: features → layer probes → confound → mel-CNN ablation) or `"video_train"` (face-crop → EfficientNet-B0 → cross-dataset eval → export ckpt).

**Colab survival rules baked in:** everything persistent (features, checkpoints, results)
is written to **Google Drive** (`/content/drive/MyDrive/paper1/`) so a disconnect costs
nothing; downloads and crops go to fast local `/content/data` (ephemeral). All stages are
resumable. Free-tier T4 ≈ Kaggle P100-ish; keep `DEBUG=True` first run, always.

In [ ]:
# ============================== SETUP ==============================
import os, sys, json, glob, math, random, time, subprocess, warnings, hashlib
from pathlib import Path
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
DATA = Path("/content/data"); DATA.mkdir(parents=True, exist_ok=True)

# --- Google Drive for persistence (skip locally) ---
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    PERSIST = Path("/content/drive/MyDrive/paper1")
else:
    PERSIST = Path("./paper1_persist")
for sub in ("features", "ckpts", "results", "figures", "reports"):
    (PERSIST / sub).mkdir(parents=True, exist_ok=True)
print("persistent store:", PERSIST)

# --- kaggle.json upload & API config ---
kag = Path.home() / ".kaggle" / "kaggle.json"
if not kag.exists():
    kag.parent.mkdir(parents=True, exist_ok=True)
    if IN_COLAB:
        print("⬆️  Upload your kaggle.json (Kaggle → Account → Create New API Token)")
        up = files.upload()                     # expects kaggle.json
        src = next((n for n in up if n.endswith(".json")), None)
        assert src, "no json uploaded"
        kag.write_bytes(up[src])
    else:
        raise SystemExit("place kaggle.json at ~/.kaggle/kaggle.json")
os.chmod(kag, 0o600)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=False)
print(subprocess.run(["kaggle", "--version"], capture_output=True, text=True).stdout or "kaggle CLI ready")

def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout: print(r.stdout[-1500:])
    if r.returncode != 0: print("STDERR:", r.stderr[-1500:])
    return r.returncode == 0

In [ ]:
# ============================== DATASETS ==============================
# Toggle what you need. SLUGS ARE MIRRORS — verify before big downloads:
  !kaggle datasets list -s "in the wild audio deepfake"
# Each entry: enabled, kind ("kaggle" | "url"), ref, dest folder under /content/data.
DATASETS = {
    # ---------- audio ----------
    "in_the_wild":   {"enabled": True,  "kind": "kaggle",
                      "ref": "bhaveshkumars/release-in-the-wild",         # matches your 14313 report path
                      "dest": "release-in-the-wild"},
    "asvspoof2019":  {"enabled": False, "kind": "kaggle",
                      "ref": "awsaf49/asvpoof-2019-dataset",              # common mirror; VERIFY slug
                      "dest": "asvspoof2019"},
    "partialspoof":  {"enabled": False, "kind": "url",                    # Zenodo record 5766198 (VERIFY file list)
                      "ref": "https://zenodo.org/api/records/5766198",
                      "dest": "partialspoof"},
    "cvoicefake":    {"enabled": False, "kind": "kaggle",
                      "ref": "YOUR-USER/YOUR-CVOICEFAKE-SLUG",            # your private upload
                      "dest": "cvoicefake"},
    # ---------- video ----------
    "celebdf_v2":    {"enabled": False, "kind": "kaggle",
                      "ref": "reubensuju/celeb-df-v2",                    # common mirror; VERIFY slug
                      "dest": "celebdf-v2"},
    "ffpp_c23":      {"enabled": False, "kind": "kaggle",
                      "ref": "YOUR-FFPP-MIRROR-SLUG",                     # search: kaggle datasets list -s faceforensics
                      "dest": "ffpp"},
    # DeepSpeak v2 / DF40 / FakeAVCeleb: LICENSED — request access, download manually,
    # upload as your own PRIVATE kaggle dataset, then add an entry here pointing at it.
}

def kaggle_download(ref, dest):
    d = DATA / dest
    if any(d.rglob("*")) if d.exists() else False:
        print(f"[skip] {dest} already populated"); return True
    d.mkdir(parents=True, exist_ok=True)
    ok = sh(f'kaggle datasets download -d "{ref}" -p "{d}" --unzip')
    if not ok: print(f"❌ {ref} failed — verify slug: kaggle datasets list -s <keywords>")
    return ok

def url_download(ref, dest):
    d = DATA / dest; d.mkdir(parents=True, exist_ok=True)
    if any(d.rglob("*")): print(f"[skip] {dest} already populated"); return True
    if "zenodo.org/api/records" in ref:      # fetch record json -> download files
        import urllib.request
        rec = json.load(urllib.request.urlopen(ref))
        for f in rec.get("files", []):
            url, name = f["links"]["self"], f["key"]
            print("→", name)
            sh(f'wget -q -c -O "{d/name}" "{url}"')
            if name.endswith((".zip", ".tar.gz", ".tgz")):
                sh(f'cd "{d}" && (unzip -qo "{name}" || tar xzf "{name}")')
        return True
    sh(f'wget -q -c -P "{d}" "{ref}"'); return True

for name, spec in DATASETS.items():
    if not spec["enabled"]: continue
    print(f"===== downloading {name} =====")
    (kaggle_download if spec["kind"] == "kaggle" else url_download)(spec["ref"], spec["dest"])

In [ ]:
# ============================== STAGE + CONFIG ==============================
STAGE = "week1_audio"        # "week1_audio" | "video_train"
DEBUG = True                 # ALWAYS smoke-test first

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.40", "soundfile", "librosa", "timm",
                "opencv-python-headless", "facenet-pytorch", "scipy", "scikit-learn"], check=False)
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import librosa, soundfile as sf
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| stage:", STAGE, "| debug:", DEBUG)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class CFG:
    SR, CROP_SEC = 16000, 4.0
    # ---- week1_audio ----
    SOURCE_NAME = "cvoicefake"
    SOURCE_REAL = str(DATA / "cvoicefake/**/real/**/*.wav")
    SOURCE_FAKE = str(DATA / "cvoicefake/**/fake/**/*.wav")
    TARGET_NAME = "in_the_wild"
    TARGET_REAL = str(DATA / "release-in-the-wild/**/real/**/*.wav")
    TARGET_FAKE = str(DATA / "release-in-the-wild/**/fake/**/*.wav")
    SRC_PER_CLS, TGT_PER_CLS = 10000, 4000
    SSL_MODELS = {"base": "facebook/wav2vec2-base", "xlsr": "facebook/wav2vec2-xls-r-300m"}
    SSL_BATCH  = 12                       # T4-safe (drop to 6 if OOM on xlsr)
    SEEDS      = [0, 1, 2, 3, 4]
    N_MELS, CNN_EPOCHS, CNN_BATCH, CNN_LR = 80, 8, 64, 3e-4
    CNN_SEEDS  = [0, 1, 2]
    # ---- video_train ----
    TRAIN_REAL = str(DATA / "ffpp/**/original*/**/*.mp4")
    TRAIN_FAKE = str(DATA / "ffpp/**/manipulated*/**/*.mp4")
    CROSS_REAL = str(DATA / "celebdf-v2/**/Celeb-real/**/*.mp4")
    CROSS_FAKE = str(DATA / "celebdf-v2/**/Celeb-synthesis/**/*.mp4")
    FRAMES_PER_VID, IMG, MARGIN = 10, 224, 0.30
    MAX_TRAIN_VIDS, MAX_CROSS_VIDS, VAL_FRAC = 1500, 400, 0.15
    V_EPOCHS, V_BATCH, V_LR, V_SEEDS = 6, 32, 2e-4, [0]   # T4: batch 32
    N_BOOT = 1000

if DEBUG:
    CFG.SRC_PER_CLS, CFG.TGT_PER_CLS = 150, 150
    CFG.SEEDS, CFG.CNN_SEEDS, CFG.CNN_EPOCHS = [0, 1], [0], 2
    CFG.MAX_TRAIN_VIDS, CFG.MAX_CROSS_VIDS = 40, 20
    CFG.FRAMES_PER_VID, CFG.V_EPOCHS = 4, 2
    CFG.N_BOOT = 200

def load_clip(path, sr=CFG.SR, crop=CFG.CROP_SEC):
    try:
        y, fsr = sf.read(path, dtype="float32", always_2d=False)
        if y.ndim > 1: y = y.mean(1)
        if fsr != sr: y = librosa.resample(y, orig_sr=fsr, target_sr=sr)
    except Exception:
        y, _ = librosa.load(path, sr=sr, mono=True)
    T = int(sr * crop)
    y = y[(len(y)-T)//2:(len(y)-T)//2+T] if len(y) >= T else np.pad(y, (0, T-len(y)))
    m = np.abs(y).max(); return y/m if m > 0 else y

def bootstrap_auc_ci(y, s, n=CFG.N_BOOT, seed=0):
    rng = np.random.default_rng(seed); y, s = np.asarray(y), np.asarray(s); v = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        v.append(roc_auc_score(y[i], s[i]))
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]

## STAGE A — `week1_audio` (S1 features → S2 layer probes → S3 confounds → S4 mel-CNN ablation)

In [ ]:
if STAGE == "week1_audio":
    def manifest(rg, fg, n_per, name, seed=1234):
        R = sorted(glob.glob(rg, recursive=True)); Fk = sorted(glob.glob(fg, recursive=True))
        assert R and Fk, f"[{name}] fix globs (real={len(R)}, fake={len(Fk)}) — enable the dataset above"
        rng = random.Random(seed); rng.shuffle(R); rng.shuffle(Fk)
        R, Fk = R[:n_per], Fk[:n_per]
        paths = R + Fk; labels = [0]*len(R) + [1]*len(Fk)
        idx = list(range(len(paths))); rng.shuffle(idx)
        print(f"[{name}] real={len(R)} fake={len(Fk)}")
        return {"paths": [paths[i] for i in idx], "labels": [labels[i] for i in idx]}

    SRC = manifest(CFG.SOURCE_REAL, CFG.SOURCE_FAKE, CFG.SRC_PER_CLS, "source")
    TGT = manifest(CFG.TARGET_REAL, CFG.TARGET_FAKE, CFG.TGT_PER_CLS, "target")
    cut = int(0.85 * len(SRC["paths"]))
    SRC_TR = {"paths": SRC["paths"][:cut], "labels": SRC["labels"][:cut]}
    SRC_VA = {"paths": SRC["paths"][cut:], "labels": SRC["labels"][cut:]}

    from transformers import Wav2Vec2Model
    @torch.no_grad()
    def extract(mk, hf, man, split):
        fx = PERSIST / "features" / f"{mk}__{split}.npy"
        fy = PERSIST / "features" / f"{mk}__{split}__y.npy"
        if fx.exists(): print(f"[cache] {fx.name}"); return fx, fy
        model = Wav2Vec2Model.from_pretrained(hf).to(DEVICE).eval().half()
        X, Y, B, BL = [], [], [], []
        def flush():
            if not B: return
            x = torch.tensor(np.stack(B), dtype=torch.float16, device=DEVICE)
            hs = model(x, output_hidden_states=True).hidden_states
            X.append(torch.stack([h.mean(1) for h in hs], 1).cpu().numpy().astype(np.float16))
            Y.extend(BL); B.clear(); BL.clear()
        for i, (p, l) in enumerate(zip(man["paths"], man["labels"])):
            B.append(load_clip(p)); BL.append(l)
            if len(B) == CFG.SSL_BATCH: flush()
            if (i+1) % 500 == 0: print(f"  [{mk}:{split}] {i+1}/{len(man['paths'])}")
        flush()
        np.save(fx, np.concatenate(X)); np.save(fy, np.array(Y, np.int8))
        del model; torch.cuda.empty_cache(); return fx, fy

    FE = {}
    for mk, hf in CFG.SSL_MODELS.items():
        for sp, mn in [("src_tr", SRC_TR), ("src_va", SRC_VA), ("tgt", TGT)]:
            FE[(mk, sp)] = extract(mk, hf, mn, sp)

    # ---- S2: layer-wise probes ----
    probe, bank = {}, {}
    for mk in CFG.SSL_MODELS:
        Xtr = np.load(FE[(mk, "src_tr")][0]).astype(np.float32); ytr = np.load(FE[(mk, "src_tr")][1])
        Xva = np.load(FE[(mk, "src_va")][0]).astype(np.float32); yva = np.load(FE[(mk, "src_va")][1])
        Xte = np.load(FE[(mk, "tgt")][0]).astype(np.float32);    yte = np.load(FE[(mk, "tgt")][1])
        probe[mk] = {}
        for L in range(Xtr.shape[1]):
            probe[mk][L] = {}
            for sd in CFG.SEEDS:
                set_seed(sd)
                sc = StandardScaler().fit(Xtr[:, L])
                clf = LogisticRegression(max_iter=2000, random_state=sd).fit(sc.transform(Xtr[:, L]), ytr)
                sv = clf.predict_proba(sc.transform(Xva[:, L]))[:, 1]
                st = clf.predict_proba(sc.transform(Xte[:, L]))[:, 1]
                probe[mk][L][sd] = {"in": float(roc_auc_score(yva, sv)), "cross": float(roc_auc_score(yte, st))}
                bank[(mk, L, sd)] = st
            print(f"[S2] {mk} L{L:02d} cross={np.mean([probe[mk][L][s]['cross'] for s in CFG.SEEDS]):.3f}")
        del Xtr, Xva, Xte
    json.dump(probe, open(PERSIST / "results" / "probe_results.json", "w"), indent=2, default=str)
    yte = np.load(FE[("base", "tgt")][1])

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for j, mk in enumerate(["base", "xlsr"]):
        Ls = sorted(probe[mk])
        for split, c in [("in", "tab:gray"), ("cross", "tab:red")]:
            mu = [np.mean([probe[mk][L][s][split] for s in CFG.SEEDS]) for L in Ls]
            sd_ = [np.std([probe[mk][L][s][split] for s in CFG.SEEDS]) for L in Ls]
            ax[j].plot(Ls, mu, "-o", ms=3, color=c, label=split)
            ax[j].fill_between(Ls, np.array(mu)-sd_, np.array(mu)+sd_, color=c, alpha=0.2)
        ax[j].axhline(0.5, ls="--", c="k", lw=0.8); ax[j].set_title(mk); ax[j].grid(alpha=0.3)
    ax[0].legend(); fig.suptitle("Fig.2 — layer-wise transfer")
    fig.savefig(PERSIST / "figures" / "fig2_layerwise.png", dpi=200); plt.close(fig)
    best = {mk: max(probe[mk], key=lambda L: np.mean([probe[mk][L][s]["cross"] for s in CFG.SEEDS])) for mk in probe}
    print("best layers:", best)

    # ---- S3: confounds ----
    def astats(p):
        y = load_clip(p); S = np.abs(librosa.stft(y, n_fft=1024, hop_length=256)) + 1e-10
        freqs = librosa.fft_frequencies(sr=CFG.SR, n_fft=1024)
        band = 20*np.log10(S.mean(1)); band -= band.max()
        above = np.where(band > -40)[0]
        return {"cutoff": float(freqs[above[-1]]) if len(above) else 0.0,
                "rolloff": float(np.mean(librosa.feature.spectral_rolloff(S=S, sr=CFG.SR, roll_percent=0.95))),
                "flatness": float(np.mean(librosa.feature.spectral_flatness(S=S)))}
    def cstats(man, cap=1200):
        rng = random.Random(7); idx = list(range(len(man["paths"]))); rng.shuffle(idx)
        return [dict(astats(man["paths"][i]), label=man["labels"][i]) for i in idx[:cap]]
    scs, tcs = cstats(SRC), cstats(TGT)
    summ = {}
    for k in ("cutoff", "rolloff", "flatness"):
        g = lambda st: np.mean([r[k] for r in st if r["label"] == 1]) - np.mean([r[k] for r in st if r["label"] == 0])
        summ[k] = {"src_gap": float(g(scs)), "tgt_gap": float(g(tcs)),
                   "sign_flip": bool(np.sign(g(scs)) != np.sign(g(tcs)))}
    json.dump(summ, open(PERSIST / "results" / "confound_summary.json", "w"), indent=2)
    print("[S3]", json.dumps(summ, indent=2))

    # ---- S4: mel-CNN ± channel augmentation ----
    MEL = librosa.filters.mel(sr=CFG.SR, n_fft=1024, n_mels=CFG.N_MELS)
    def logmel(y):
        M = np.log(MEL @ (np.abs(librosa.stft(y, n_fft=1024, hop_length=256))**2) + 1e-8)
        return ((M - M.mean())/(M.std()+1e-6)).astype(np.float32)
    def chan_aug(y, rng):
        y = y.copy()
        if rng.random() < 0.8:
            snr = rng.uniform(8, 30)
            n = np.random.default_rng(rng.randint(0, 1<<30)).standard_normal(len(y)).astype(np.float32)
            n = np.convolve(n, np.array([1.0, rng.uniform(-0.9, 0.9)], np.float32), "same")
            n *= math.sqrt((np.mean(y**2)+1e-12)/(np.mean(n**2)+1e-12)/(10**(snr/10))); y = y + n
        if rng.random() < 0.7:
            Y = librosa.stft(y, n_fft=1024, hop_length=256)
            fr = librosa.fft_frequencies(sr=CFG.SR, n_fft=1024)
            Y[fr > rng.uniform(3000, 7600), :] *= rng.uniform(0, 0.1)
            y = librosa.istft(Y, hop_length=256, length=len(y))
        if rng.random() < 0.3: y = np.clip(y, -rng.uniform(0.6, 0.95), rng.uniform(0.6, 0.95))
        y *= rng.uniform(0.5, 1.0); m = np.abs(y).max(); return (y/m if m > 0 else y).astype(np.float32)
    class MelDS(torch.utils.data.Dataset):
        def __init__(self, man, aug=False, seed=0): self.m, self.aug, self.seed = man, aug, seed
        def __len__(self): return len(self.m["paths"])
        def __getitem__(self, i):
            y = load_clip(self.m["paths"][i])
            if self.aug: y = chan_aug(y, random.Random(self.seed*1_000_003 + i))
            return torch.from_numpy(logmel(y))[None], self.m["labels"][i]
    class MelCNN(nn.Module):
        def __init__(self):
            super().__init__(); ch = [1, 32, 64, 128, 128]
            self.b = nn.Sequential(*[nn.Sequential(nn.Conv2d(ch[i], ch[i+1], 3, padding=1),
                nn.BatchNorm2d(ch[i+1]), nn.ReLU(), nn.MaxPool2d(2)) for i in range(4)])
            self.h = nn.Linear(128, 1)
        def forward(self, x): return self.h(self.b(x).mean((2, 3))).squeeze(-1)
    @torch.no_grad()
    def cnn_eval(model, man):
        dl = torch.utils.data.DataLoader(MelDS(man), batch_size=CFG.CNN_BATCH, num_workers=2)
        S, Y = [], []
        for x, y in dl:
            S.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()); Y.append(y.numpy())
        return np.concatenate(S), np.concatenate(Y)
    abl = {"no_aug": {}, "chan_aug": {}}
    for sd in CFG.CNN_SEEDS:
        for aug in (False, True):
            set_seed(sd); model = MelCNN().to(DEVICE)
            opt = torch.optim.AdamW(model.parameters(), lr=CFG.CNN_LR)
            dl = torch.utils.data.DataLoader(MelDS(SRC_TR, aug, sd), batch_size=CFG.CNN_BATCH,
                                             shuffle=True, num_workers=2, drop_last=True)
            for ep in range(CFG.CNN_EPOCHS):
                model.train()
                for x, y in dl:
                    loss = F.binary_cross_entropy_with_logits(model(x.to(DEVICE)), y.float().to(DEVICE))
                    opt.zero_grad(); loss.backward(); opt.step()
            sv, yv = cnn_eval(model, SRC_VA); st, yt = cnn_eval(model, TGT)
            abl["chan_aug" if aug else "no_aug"][sd] = {
                "in": float(roc_auc_score(yv, sv)), "cross": float(roc_auc_score(yt, st))}
            print(f"[S4] aug={aug} seed={sd} in={abl['chan_aug' if aug else 'no_aug'][sd]['in']:.3f} "
                  f"cross={abl['chan_aug' if aug else 'no_aug'][sd]['cross']:.3f}")
    json.dump(abl, open(PERSIST / "results" / "ablation.json", "w"), indent=2)

    print("="*66, "\nWEEK1 (Colab) VERDICT")
    for mk in best:
        s0 = bank[(mk, best[mk], CFG.SEEDS[0])]
        print(f"  {mk} best L{best[mk]:02d} cross-AUC={roc_auc_score(yte, s0):.3f} CI={bootstrap_auc_ci(yte, s0)}")
    for cond in abl:
        cs = [abl[cond][s]["cross"] for s in abl[cond]]
        print(f"  mel-CNN {cond}: cross {np.mean(cs):.3f} ± {np.std(cs):.3f}")
    print("  flips:", [k for k, v in summ.items() if v["sign_flip"]])
    print("  artifacts →", PERSIST)

## STAGE B — `video_train` (face crops → EfficientNet-B0 → cross-dataset → export ckpt)

In [ ]:
if STAGE == "video_train":
    import cv2, timm
    from facenet_pytorch import MTCNN
    mtcnn = MTCNN(keep_all=False, select_largest=True, post_process=False, device=DEVICE)
    CROPS = Path("/content/data/crops"); CROPS.mkdir(parents=True, exist_ok=True)
    IMN_M = np.array([0.485, 0.456, 0.406], np.float32); IMN_S = np.array([0.229, 0.224, 0.225], np.float32)

    def crop_video(path, out_dir, vid):
        if len(list(out_dir.glob(f"{vid}__*.jpg"))) >= max(1, CFG.FRAMES_PER_VID//2): return True
        cap = cv2.VideoCapture(path); tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if tot <= 0: cap.release(); return False
        ok_any = False
        for k, i in enumerate(np.linspace(0, tot-1, CFG.FRAMES_PER_VID).astype(int)):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, fr = cap.read()
            if not ok: continue
            rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
            box, prob = mtcnn.detect(rgb)
            if box is None or prob is None or prob[0] is None or prob[0] < 0.9: continue
            x0, y0, x1, y1 = box[0]; w, h = x1-x0, y1-y0
            x0 = max(int(x0-CFG.MARGIN*w), 0); y0 = max(int(y0-CFG.MARGIN*h), 0)
            x1 = min(int(x1+CFG.MARGIN*w), rgb.shape[1]); y1 = min(int(y1+CFG.MARGIN*h), rgb.shape[0])
            crop = cv2.resize(rgb[y0:y1, x0:x1], (CFG.IMG, CFG.IMG))
            cv2.imwrite(str(out_dir/f"{vid}__{k}.jpg"), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR),
                        [cv2.IMWRITE_JPEG_QUALITY, 95]); ok_any = True
        cap.release(); return ok_any

    def build(rg, fg, split, cap_n):
        man = []
        for nm, lab, g in [("real", 0, rg), ("fake", 1, fg)]:
            vs = sorted(glob.glob(g, recursive=True))
            assert vs, f"[{split}/{nm}] fix glob: {g} (enable dataset above / verify slug)"
            random.Random(0).shuffle(vs)
            od = CROPS / split / nm; od.mkdir(parents=True, exist_ok=True)
            for j, v in enumerate(vs[:cap_n]):
                if crop_video(v, od, Path(v).stem): man.append({"vid": Path(v).stem, "label": lab, "dir": str(od)})
                if (j+1) % 100 == 0: print(f"  [{split}/{nm}] {j+1}")
        print(f"[{split}] {len(man)} videos"); return man
    M_POOL = build(CFG.TRAIN_REAL, CFG.TRAIN_FAKE, "train_pool", CFG.MAX_TRAIN_VIDS)
    M_CROSS = build(CFG.CROSS_REAL, CFG.CROSS_FAKE, "cross", CFG.MAX_CROSS_VIDS)
    random.Random(42).shuffle(M_POOL)
    nv = int(len(M_POOL)*CFG.VAL_FRAC); M_VAL, M_TR = M_POOL[:nv], M_POOL[nv:]   # video-level split

    def vaug(img, rng):
        if rng.random() < 0.5: img = img[:, ::-1]
        if rng.random() < 0.5:
            _, e = cv2.imencode(".jpg", cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
                                [cv2.IMWRITE_JPEG_QUALITY, rng.randint(30, 90)])
            img = cv2.cvtColor(cv2.imdecode(e, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if rng.random() < 0.3: img = cv2.GaussianBlur(img, (rng.choice([3, 5]),)*2, 0)
        if rng.random() < 0.3:
            s = rng.uniform(0.4, 0.9); h, w = img.shape[:2]
            img = cv2.resize(cv2.resize(img, (int(w*s), int(h*s))), (w, h))
        return np.ascontiguousarray(img)
    class FDS(torch.utils.data.Dataset):
        def __init__(self, man, train=False):
            self.items = [(str(f), m["label"], m["vid"]) for m in man
                          for f in Path(m["dir"]).glob(f"{m['vid']}__*.jpg")]
            self.train = train
        def __len__(self): return len(self.items)
        def __getitem__(self, i):
            p, l, v = self.items[i]
            img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
            if self.train: img = vaug(img, random.Random(random.getrandbits(32)))
            return torch.from_numpy(((img.astype(np.float32)/255.-IMN_M)/IMN_S).transpose(2, 0, 1)), l, v
    @torch.no_grad()
    def veval(model, ds, name):
        model.eval(); dl = torch.utils.data.DataLoader(ds, batch_size=CFG.V_BATCH, num_workers=2)
        S, Y, V = [], [], []
        for x, y, v in dl:
            S.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()); Y.append(y.numpy()); V.extend(v)
        s, y = np.concatenate(S), np.concatenate(Y)
        agg = {}
        for sc, lb, vd in zip(s, y, V): agg.setdefault(vd, {"s": [], "y": int(lb)})["s"].append(float(sc))
        cy = [a["y"] for a in agg.values()]; cs = [float(np.mean(a["s"])) for a in agg.values()]
        r = {"frame_auc": float(roc_auc_score(y, s)), "clip_auc": float(roc_auc_score(cy, cs)),
             "ci": bootstrap_auc_ci(cy, cs)}
        print(f"[eval:{name}] frame={r['frame_auc']:.4f} clip={r['clip_auc']:.4f} CI={r['ci']}")
        return r

    out = {}
    for sd in CFG.V_SEEDS:
        set_seed(sd)
        model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=1).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=CFG.V_LR, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.V_EPOCHS)
        dl = torch.utils.data.DataLoader(FDS(M_TR, True), batch_size=CFG.V_BATCH,
                                         shuffle=True, num_workers=2, drop_last=True)
        best = 0.0
        ck = PERSIST / "ckpts" / f"video_effb0_v18plus_seed{sd}.pt"
        for ep in range(CFG.V_EPOCHS):
            model.train(); tot = 0.0; t0 = time.time()
            for x, y, _ in dl:
                yv = y.float().to(DEVICE)*0.95 + 0.025
                loss = F.binary_cross_entropy_with_logits(model(x.to(DEVICE)).squeeze(-1), yv)
                opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
            sched.step()
            r = veval(model, FDS(M_VAL), f"val sd{sd} ep{ep+1}")
            print(f"  [sd{sd} ep{ep+1}] loss={tot/len(dl):.4f} ({time.time()-t0:.0f}s)")
            if r["clip_auc"] > best: best = r["clip_auc"]; torch.save(model.state_dict(), ck)
        model.load_state_dict(torch.load(ck, map_location=DEVICE))
        out[sd] = {"in_domain": veval(model, FDS(M_VAL), f"IN sd{sd}"),
                   "cross":     veval(model, FDS(M_CROSS), f"CROSS sd{sd}")}
        h = hashlib.sha256(open(ck, "rb").read()).hexdigest()
        out[sd]["sha256"] = h
        print(f"  ckpt {ck.name} sha256={h[:16]}…  (saved to Drive)")
    json.dump(out, open(PERSIST / "results" / "video_train_results.json", "w"), indent=2)
    print("checkpoints + results persisted →", PERSIST)

print("\nDone. STAGE =", STAGE, "| DEBUG =", DEBUG,
      "\nAll persistent artifacts:", PERSIST,
      "\nNext: flip DEBUG=False for the real run; switch STAGE for the other pipeline.")